# HW02 — From Logits to Language

**Central question:** The model produces only a probability distribution; what decides which token is actually selected?

Every graded implementation lives only in `src/*.py`. The notebook does not redefine those functions.

In [1]:
%load_ext autoreload
%autoreload 2

## Section 0 — Environment, model, and reproducibility

### Goal
Record the environment and load the model safely on the CPU

### Concept
Keeping the model, device, and seed fixed is required for a fair comparison.

### Your task
Verify the assets and load the model.

### Constraints
Load the checkpoint only with `weights_only=True`; all parameters must remain on the CPU.

### Hint
The parameter count and checksum must match the published values.


In [2]:
import json
import math
import platform
import time
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import torch

from src.config import count_parameters, set_seed
from src.model_io import load_pretrained_model, verify_assets
from src.tokenizer import EOS_TOKEN_ID, encode_text, token_display

PROJECT = Path.cwd()
if not (PROJECT / "src").is_dir():
    raise RuntimeError("Notebook must run from the package root")
(PROJECT / "figures").mkdir(parents=True, exist_ok=True)
(PROJECT / "results").mkdir(parents=True, exist_ok=True)

SEED = 1403
set_seed(SEED)
torch.set_num_threads(max(1, min(4, psutil.cpu_count(logical=False) or 1)))
assets = verify_assets(PROJECT / "assets")
PROMPTS = assets["prompts"]
model = load_pretrained_model(PROJECT / "assets").cpu()
model.eval()
assert all(parameter.device.type == "cpu" for parameter in model.parameters())
environment = {
    "python_version": platform.python_version(), "torch_version": torch.__version__,
    "device": "cpu", "cuda_available": torch.cuda.is_available(),
    "parameter_count": count_parameters(model), "checkpoint_sha256": assets["checkpoint_sha256"],
    "context_length": model.config.context_length, "vocabulary_size": model.config.vocab_size, "seed": SEED,
    "cpu": platform.processor() or platform.machine(),
}
assert environment["parameter_count"] == 644736
(PROJECT / "results/environment.json").write_text(json.dumps(environment, indent=2) + "\n", encoding="utf-8")
print({key: environment[key] for key in ["python_version", "torch_version", "device", "cuda_available", "parameter_count"]})

{'python_version': '3.15.2', 'torch_version': '2.15.0+cpu', 'device': 'cpu', 'cuda_available': False, 'parameter_count': 644736}


## Section 1 — Inspecting logits and next-token candidates

### Goal
Inspect the model's raw output and convert it to a probability distribution

### Concept
The model produces one logit for every vocabulary token; a logit is not a probability.

### Your task
Run your `stable_softmax` on the logits at the final position.

### Constraints
The TODO function is not redefined in the notebook, and the display is limited to 15 tokens.

### Hint
Subtract the maximum before applying `exp`.

In [3]:
from src.plotting import plot_next_token_distribution
from src.probability import stable_softmax

inspection_prompt = PROMPTS[0]
prompt_ids = torch.tensor([encode_text(inspection_prompt["text"], add_bos=True)], dtype=torch.long)
with torch.inference_mode():
    next_logits = model(prompt_ids).logits[:, -1, :][0]
probabilities = stable_softmax(next_logits)
assert probabilities.shape == next_logits.shape == (260,)
assert abs(float(probabilities.sum()) - 1) < 1e-6
ordered = sorted(range(260), key=lambda i: (-float(probabilities[i]), i))[:15]
rows = []
cumulative = 0.0
for rank, token_id in enumerate(ordered, 1):
    cumulative += float(probabilities[token_id])
    rows.append({"rank": rank, "token_id": token_id, "token_display": token_display(token_id), "logit": float(next_logits[token_id]), "probability": float(probabilities[token_id]), "cumulative_probability": cumulative})
top_tokens = pd.DataFrame(rows)
top_tokens.to_csv(PROJECT / "results/top_tokens.csv", index=False)
plot_next_token_distribution(top_tokens, PROJECT / "figures/next_token_distribution.png")
print("logit shape:", tuple(next_logits.shape), "probability sum:", round(float(probabilities.sum()), 7))
display(top_tokens)

logit shape: (260,) probability sum: 1.0


,rank,token_id,token_display,logit,probability,cumulative_probability
0,1,36,,3.103769,0.100320,0.100320
1,2,50,.,2.737130,0.069528,0.169847
2,3,48,",",2.619825,0.061832,0.231679
3,4,52,0,2.322595,0.045933,0.277612
4,5,67,?,2.028626,0.034234,0.311846
5,6,230,�,2.001879,0.033330,0.345177
6,7,105,e,1.906246,0.030291,0.375467
7,8,49,-,1.822038,0.027844,0.403312
8,9,37,!,1.442563,0.019052,0.422363
9,10,167,�,1.355593,0.017465,0.439828


## Section 2 — Numerical stability of softmax

### Goal
Observe overflow in a naive implementation

### Concept
Adding or subtracting a shared constant does not change probability ratios.

### Your task
Write the naive method only for this experiment and call `stable_softmax` from `src`.

### Constraints
Using `torch.softmax` or `F.softmax` inside `stable_softmax` is prohibited.

### Hint
The numerator and denominator of softmax are both multiplied by `exp(-c)`.

In [7]:
def stable_softmax(logits: torch.Tensor) -> torch.Tensor:
    """Textbook softmax without max-shift. For demonstration only."""
    exps = torch.exp(logits)
    return exps / exps.sum(dim=-1, keepdim=True)


extreme_logits = torch.tensor([1000.0, 1001.0, 1002.0])
naive = torch.exp(extreme_logits) / torch.exp(extreme_logits).sum()
stable = stable_softmax(extreme_logits)
stability = {"input_logits": extreme_logits.tolist(), "naive_output_finite": bool(torch.isfinite(naive).all()), "stable_probabilities": stable.tolist(), "stable_probability_sum": float(stable.sum())}
(PROJECT / "results/softmax_stability.json").write_text(json.dumps(stability, indent=2) + "\n", encoding="utf-8")
print(stability)

{'input_logits': [1000.0, 1001.0, 1002.0], 'naive_output_finite': False, 'stable_probabilities': [nan, nan, nan], 'stable_probability_sum': nan}


## Section 3 — Temperature and entropy

### Goal
Measure how temperature changes the distribution's shape

### Concept
Dividing logits by positive T preserves rank order while changing their effective gaps.

### Your task
Compare four temperatures and calculate natural-log entropy and 90% support size.

### Constraints
Calling the result 'more creative' without a numerical measure is not sufficient.

### Hint
Use natural logarithms for entropy and the limit `p log p = 0` at zero.

In [9]:
from src.plotting import plot_temperature_distributions
from src.probability import apply_temperature, distribution_entropy, support_size_for_mass

temperature_rows, plot_rows = [], []
base_order = sorted(range(260), key=lambda i: (-float(next_logits[i]), i))
for temperature in [0.5, 1.0, 1.5, 2.0]:
    current = stable_softmax(apply_temperature(next_logits, temperature))
    current_order = sorted(range(260), key=lambda i: (-float(current[i]), i))
    temperature_rows.append({"temperature": temperature, "top_token_probability": float(current[current_order[0]]), "entropy_nats": float(distribution_entropy(current)), "support_size_90": support_size_for_mass(current, 0.90), "rank_order_consistent": current_order == base_order, "probability_sum": float(current.sum())})
    for rank, token_id in enumerate(current_order[:15], 1):
        plot_rows.append({"temperature": temperature, "rank": rank, "probability": float(current[token_id])})
temperature_metrics = pd.DataFrame(temperature_rows)
temperature_metrics.to_csv(PROJECT / "results/temperature_metrics.csv", index=False)
plot_temperature_distributions(pd.DataFrame(plot_rows), PROJECT / "figures/temperature_distributions.png")
display(temperature_metrics)

,temperature,top_token_probability,entropy_nats,support_size_90,rank_order_consistent,probability_sum
0,0.5,0.365772,2.250245,8,True,1.0
1,1.0,0.100320,4.614167,180,True,1.0
2,1.5,0.040660,5.250090,209,True,1.0
3,2.0,0.023774,5.417492,217,True,1.0


## Section 4 — Greedy versus second-best

### Goal
Show that rank two does not represent a fixed amount of uncertainty

### Concept
Greedy selects rank one; the deliberately defective policy always selects rank two.

### Your task
Run both selectors on the model and on two hand-built distributions.

### Constraints
Break ties in favor of the smaller token ID.

### Hint
Use `log` to convert hand-built probabilities to logits.

In [ ]:
from src.decoding import select_greedy, select_second_best


def compact_choice(result):
    return {"selected_token_id": result.token_id, "rank": result.rank, "model_probability": result.model_probability, "sampling_probability": result.sampling_probability}

deterministic = {"model": {"greedy": compact_choice(select_greedy(next_logits)), "second_best": compact_choice(select_second_best(next_logits))}}
for label, distribution in {"distribution_A": [0.51, 0.48, 0.01], "distribution_B": [0.95, 0.03, 0.02]}.items():
    logits = torch.log(torch.tensor(distribution))
    deterministic[label] = {"probabilities": distribution, "greedy": compact_choice(select_greedy(logits)), "second_best": compact_choice(select_second_best(logits))}
(PROJECT / "results/deterministic_choices.json").write_text(json.dumps(deterministic, indent=2) + "\n", encoding="utf-8")
print(json.dumps(deterministic, indent=2))

## Section 5 — Multinomial, top-k, and top-p

### Goal
Build and measure candidate sets

### Concept
Top-k retains a fixed count; top-p retains the smallest support that covers the probability mass.

### Your task
Run five policies with fixed seeds and record retained mass.

### Constraints
Renormalize after filtering; do not equate model probability with sampling probability.

### Hint
`SelectionResult` exposes `candidate_count` and `sampling_probability`.

In [12]:
from src.decoding import sample_multinomial, sample_top_k, sample_top_p

sampling_specs = [
    ("multinomial", "full", lambda g: sample_multinomial(next_logits, g)),
    ("top_k", "5", lambda g: sample_top_k(next_logits, 5, g)),
    ("top_k", "20", lambda g: sample_top_k(next_logits, 20, g)),
    ("top_p", "0.80", lambda g: sample_top_p(next_logits, 0.80, g)),
    ("top_p", "0.95", lambda g: sample_top_p(next_logits, 0.95, g)),
]
sampling_rows = []
scaled = stable_softmax(next_logits)
ordered_all = sorted(range(260), key=lambda i: (-float(scaled[i]), i))
for offset, (strategy, parameter, call) in enumerate(sampling_specs):
    seed = SEED + offset
    result = call(torch.Generator(device="cpu").manual_seed(seed))
    retained_mass = float(scaled[ordered_all[:result.candidate_count]].sum())
    sampling_rows.append({"strategy": strategy, "parameter": parameter, "candidate_count": result.candidate_count, "retained_probability_mass": retained_mass, "selected_token_id": result.token_id, "selected_token_display": token_display(result.token_id), "model_probability": result.model_probability, "sampling_probability": result.sampling_probability, "rank": result.rank, "seed": seed})
sampling_candidates = pd.DataFrame(sampling_rows)
sampling_candidates.to_csv(PROJECT / "results/sampling_candidate_sets.csv", index=False)
display(sampling_candidates)

,strategy,parameter,candidate_count,retained_probability_mass,selected_token_id,selected_token_display,model_probability,sampling_probability,rank,seed
0,multinomial,full,260,1.000000,36,,0.100320,0.100320,1,1403
1,top_k,5,5,0.311846,52,0,0.045933,0.147294,4,1404
2,top_k,20,20,0.533286,170,�,0.007690,0.014421,17,1405
3,top_p,0.80,123,0.800842,36,,0.100320,0.125268,1,1406
4,top_p,0.95,212,0.950374,230,�,0.033330,0.035071,6,1407


## Section 6 — Manual generation loop

### Goal
Run sequential inference without `generate`

### Concept
Each model pass provides only the final-position logits used to select one token.

### Your task
Run four policies with `generate_tokens` and combine their traces.

### Constraints
`model.generate` and high-level wrappers are prohibited; each iteration appends exactly one token.

### Hint
Create one CPU `torch.Generator` and crop context from the left.

In [ ]:
from src.generation import generate_tokens

generation_prompt = PROMPTS[1]
generation_prompt_ids = torch.tensor([encode_text(generation_prompt["text"], add_bos=True)], dtype=torch.long)
selectors = {
    "greedy": lambda logits, generator: select_greedy(logits),
    "second_best": lambda logits, generator: select_second_best(logits),
    "top_k_20": lambda logits, generator: sample_top_k(logits, 20, generator),
    "top_p_0.95": lambda logits, generator: sample_top_p(logits, 0.95, generator),
}
trace_rows = []
for run_index, (name, selector) in enumerate(selectors.items()):
    result = generate_tokens(model, generation_prompt_ids, selector, 24, EOS_TOKEN_ID, 128, SEED)
    assert result.forward_passes == len(result.generated_token_ids) == len(result.trace)
    for row in result.trace:
        trace_rows.append({"run_id": f"manual_{run_index}", "prompt_id": generation_prompt["id"], "strategy": name, "seed": SEED, **row})
same_a = generate_tokens(model, generation_prompt_ids, selectors["top_p_0.95"], 12, EOS_TOKEN_ID, 128, 77)
same_b = generate_tokens(model, generation_prompt_ids, selectors["top_p_0.95"], 12, EOS_TOKEN_ID, 128, 77)
different = generate_tokens(model, generation_prompt_ids, selectors["top_p_0.95"], 12, EOS_TOKEN_ID, 128, 78)
assert same_a.generated_token_ids == same_b.generated_token_ids
seed_difference_observed = same_a.generated_token_ids != different.generated_token_ids
generation_trace = pd.DataFrame(trace_rows)
generation_trace.to_csv(PROJECT / "results/generation_trace.csv", index=False)
print("trace rows:", len(generation_trace), "different-seed difference observed:", seed_difference_observed)
display(generation_trace.head(8))

## Section 7 — Deliberately defective second-best decoding

### Goal
Compare greedy with always choosing rank two under controlled conditions

### Concept
More diversity is not necessarily controlled or more probable.

### Your task
Measure rank, model probability, entropy, cumulative log-probability, and repetition at every step.

### Constraints
Do not assume in advance that second-best is worse on every individual metric.

### Hint
Measure likelihood with `model_probability`, not `sampling_probability`.

In [ ]:
from src.metrics import token_metrics
from src.plotting import plot_second_best_decay

comparison_rows = []
for name in ["greedy", "second_best"]:
    result = generate_tokens(model, generation_prompt_ids, selectors[name], 32, EOS_TOKEN_ID, 128, SEED)
    cumulative = 0.0
    prefix = []
    for row, token_id in zip(result.trace, result.generated_token_ids, strict=True):
        cumulative += math.log(float(row["model_probability"]))
        prefix.append(token_id)
        comparison_rows.append({"strategy": name, "step": row["step"], "rank": row["rank"], "model_probability": row["model_probability"], "entropy_nats": row["entropy_nats"], "cumulative_sequence_log_probability": cumulative, "adjacent_repeat_rate": token_metrics(prefix)["adjacent_repeat_rate"]})
second_best_comparison = pd.DataFrame(comparison_rows)
second_best_comparison.to_csv(PROJECT / "results/second_best_comparison.csv", index=False)
plot_second_best_decay(second_best_comparison, PROJECT / "figures/second_best_probability_decay.png")
display(second_best_comparison.groupby("strategy").tail(1))

## Section 8 — Local probability versus global sequence probability

### Goal
Construct an exact counterexample to local selection

### Concept
A sequence log-probability is the sum of its conditional log-probabilities.

### Your task
Run greedy and exhaustive search on the provided tree asset.

### Constraints
Calculate in log space and reject invalid transition probabilities.

### Hint
Every depth-two path can be enumerated here; this is not feasible for an LLM.

In [ ]:
from src.sequence_search import exhaustive_best_sequence, greedy_tree_sequence

tree = assets["tree"]
toy = {"greedy": greedy_tree_sequence(tree, tree["start_state"], tree["depth"]), "optimal": exhaustive_best_sequence(tree, tree["start_state"], tree["depth"])}
assert toy["greedy"]["tokens"] != toy["optimal"]["tokens"]
assert toy["optimal"]["sequence_probability"] > toy["greedy"]["sequence_probability"]
(PROJECT / "results/toy_sequence_search.json").write_text(json.dumps(toy, indent=2) + "\n", encoding="utf-8")
print(toy)

## Section 9 — Controlled decoding benchmark

### Goal
Compare policies while holding the model, prompts, and token budget fixed

### Concept
Diversity and likelihood metrics capture different, limited aspects of output.

### Your task
Run every required prompt and policy, then aggregate the metrics.

### Constraints
Exact prose and wall-clock time are not graded; model, length, tokenizer, and seed policy remain fixed.

### Hint
Use only `model_probability` when computing likelihood comparisons.

In [ ]:
from src.metrics import benchmark_metric_row, trace_metrics
from src.plotting import plot_diversity_vs_probability, plot_entropy_by_step

benchmark_specs = {
    "greedy": (False, lambda logits, g: select_greedy(logits)),
    "second_best": (False, lambda logits, g: select_second_best(logits)),
    "multinomial_T1.0": (True, lambda logits, g: sample_multinomial(logits, g, 1.0)),
    "top_k_5_T1.0": (True, lambda logits, g: sample_top_k(logits, 5, g, 1.0)),
    "top_k_20_T1.0": (True, lambda logits, g: sample_top_k(logits, 20, g, 1.0)),
    "top_p_0.80_T1.0": (True, lambda logits, g: sample_top_p(logits, 0.80, g, 1.0)),
    "top_p_0.95_T1.0": (True, lambda logits, g: sample_top_p(logits, 0.95, g, 1.0)),
    "temperature_0.5_sampling": (True, lambda logits, g: sample_multinomial(logits, g, 0.5)),
    "temperature_1.5_sampling": (True, lambda logits, g: sample_multinomial(logits, g, 1.5)),
}
generation_rows, metric_rows, benchmark_trace_rows = [], [], []
benchmark_started = time.perf_counter()
for prompt in PROMPTS:
    ids = torch.tensor([encode_text(prompt["text"], add_bos=True)], dtype=torch.long)
    for strategy, (stochastic_policy, selector) in benchmark_specs.items():
        seeds = [1403, 1404, 1405] if stochastic_policy else [1403]
        for seed in seeds:
            run_id = f'{prompt["id"]}__{strategy}__{seed}'
            result = generate_tokens(model, ids, selector, 40, EOS_TOKEN_ID, 128, seed)
            aggregate = trace_metrics(result.trace)
            generation_rows.append({"run_id": run_id, "prompt_id": prompt["id"], "category": prompt["category"], "prompt": prompt["text"], "strategy": strategy, "seed": seed, "generated_text": result.decoded_text, "generated_token_count": len(result.generated_token_ids), "stop_reason": result.stop_reason, "sequence_log_probability": aggregate["sequence_log_probability"], "average_log_probability": aggregate["average_log_probability"]})
            metric_rows.append(benchmark_metric_row(run_id, prompt["id"], strategy, seed, result.generated_token_ids, result.trace))
            for row in result.trace:
                benchmark_trace_rows.append({"run_id": run_id, "prompt_id": prompt["id"], "strategy": strategy, **row})
generations = pd.DataFrame(generation_rows)
decoding_metrics = pd.DataFrame(metric_rows)
generations.to_csv(PROJECT / "results/generations.csv", index=False)
decoding_metrics.to_csv(PROJECT / "results/decoding_metrics.csv", index=False)
plot_entropy_by_step(pd.DataFrame(benchmark_trace_rows), PROJECT / "figures/entropy_by_step.png")
plot_diversity_vs_probability(decoding_metrics, PROJECT / "figures/diversity_vs_probability.png")
benchmark_runtime_seconds = time.perf_counter() - benchmark_started
print("runs:", len(generations), "runtime seconds:", round(benchmark_runtime_seconds, 2))
display(decoding_metrics.groupby("strategy")[["mean_selected_model_probability", "mean_selected_rank", "token_distinct_2"]].mean().round(3))

## Section 10 — Entropy-adaptive decoder

### Goal
Choose a threshold on calibration data and evaluate it on held-out data

### Concept
The heuristic uses greedy below the threshold and top-p above it.

### Your task
Measure calibration-prompt entropy, choose the median, and evaluate on separate prompts.

### Constraints
A threshold without evidence is invalid; evaluation prompts must not appear in calibration.

### Hint
The returned strategy records the branch actually used.

In [ ]:
from src.decoding import select_entropy_adaptive

calibration, held_out = PROMPTS[:3], PROMPTS[3:]
calibration_entropies = []
for prompt in calibration:
    ids = torch.tensor([encode_text(prompt["text"], add_bos=True)], dtype=torch.long)
    with torch.inference_mode():
        logits = model(ids).logits[0, -1]
    calibration_entropies.append(float(distribution_entropy(stable_softmax(logits))))
entropy_threshold = float(np.median(calibration_entropies))
adaptive_rows = []


def adaptive_selector(logits, generator):
    return select_entropy_adaptive(logits, entropy_threshold, 0.90, generator)


for prompt in held_out:
    ids = torch.tensor([encode_text(prompt["text"], add_bos=True)], dtype=torch.long)
    result = generate_tokens(model, ids, adaptive_selector, 40, EOS_TOKEN_ID, 128, SEED)
    for row in result.trace:
        adaptive_rows.append({"prompt_id": prompt["id"], "step": row["step"], "entropy_nats": row["entropy_nats"], "entropy_threshold": entropy_threshold, "branch": "greedy" if row["strategy"].endswith("greedy") else "top_p", "selected_token_id": row["selected_token_id"], "selected_token_display": row["selected_token_display"], "model_probability": row["model_probability"], "sampling_probability": row["sampling_probability"], "rank": row["rank"], "candidate_count": row["candidate_count"]})
adaptive = pd.DataFrame(adaptive_rows)
adaptive.to_csv(PROJECT / "results/adaptive_decoder_results.csv", index=False)
print("calibration entropies:", [round(value, 4) for value in calibration_entropies], "threshold:", round(entropy_threshold, 4))
display(adaptive["branch"].value_counts(normalize=True).rename("rate"))

## Section 11 — Summary and submission package

### Goal
Build the final comparison table and run the student release gate

### Concept
No single decoder wins on every dimension.

### Your task
Save the comparison table, validate artifacts, run tests, and build the ZIP.

### Constraints
Course assets, checkpoints, mentor files, and nested ZIPs must not enter the submission.

### Hint
Complete the report with precise evidence references before packaging.

In [ ]:
from src.validation import validate_artifacts

summary_rows = [
    ["greedy", True, "highest-probability token", False, False, False, "fixed: 1", "usually low", "repetition and local myopia"],
    ["second_best", True, "always rank 2", False, False, False, "fixed: 1", "uncontrolled", "may force very unlikely tokens"],
    ["multinomial", False, "full vocabulary", True, True, True, "fixed: vocabulary", "often high", "long low-probability tail"],
    ["top-k", False, "highest k tokens", True, True, True, "fixed: k", "controlled by k", "fixed k ignores distribution shape"],
    ["top-p", False, "minimal mass-p support", True, True, True, "variable", "adapts to concentration", "threshold and seed sensitivity"],
    ["entropy-adaptive", False, "greedy or top-p by entropy", True, True, True, "1 or variable", "conditional", "calibration heuristic may not generalize"],
]
columns = ["strategy", "deterministic", "candidate_set_rule", "uses_randomness", "seed_sensitive", "can_select_low_ranked_tokens", "candidate_count_behavior", "typical_diversity_behavior", "major_failure_mode"]
strategy_summary = pd.DataFrame(summary_rows, columns=columns)
strategy_summary.to_csv(PROJECT / "results/decoding_strategy_summary.csv", index=False)
artifact_summary = validate_artifacts(PROJECT)
print("Artifact validation passed:", artifact_summary)
print("Run public tests: uv run pytest tests -q")
print("Package: uv run python scripts/package_submission.py --student-id 123456")
display(strategy_summary)

## Final checklist

- Every TODO was completed only in `src`.
- A clean execution reproduced thirteen result files and five figures.
- Report answers cite specific rows or figures.
- Public tests and the validator were run.
- The ZIP contains no course assets or mentor content and is smaller than 20 MiB.